In [ ]:
Day 27 — Async API Resilience and Idempotency
Date: Monday, November 2, 2026
Theme: Python and APIs
Goal: Add asynchronous request handling, duplicate-request protection, concurrency limits and timeout control to the governed Finance assistant.

This extends the earlier retry work without repeating it. Databricks distinguishes client-side and server-side timeouts and recommends monitoring latency, throughput, errors and concurrency. Databricks on AWS

The SAP Cloud SDK for AI Python orchestration package also provides asynchronous execution and request-level timeout controls.

# Prepare the project

In [1]:
# DAY 27 — PROJECT SETUP
#
# Purpose:
# Locate the existing project and prepare today's data,
# application, artifact, audit and notes folders.

from pathlib import Path
from datetime import datetime, timezone
import asyncio
import hashlib
import importlib.util
import json
import sys


def locate_project_root(starting_folder: Path) -> Path:
    """
    Locate the Git project from the notebook folder.
    """

    for possible_root in [
        starting_folder,
        *starting_folder.parents,
    ]:
        has_git = (
            possible_root / ".git"
        ).exists()

        has_project_folders = (
            (possible_root / "app").exists()
            and (possible_root / "data").exists()
        )

        if has_git or has_project_folders:
            return possible_root

    return starting_folder


PROJECT_ROOT = locate_project_root(
    Path.cwd().resolve()
)

for folder_name in [
    "app",
    "artifacts",
    "audit",
    "data",
    "notes",
    "notebooks",
]:
    (PROJECT_ROOT / folder_name).mkdir(
        parents=True,
        exist_ok=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def save_json(relative_path: str, payload) -> Path:
    """
    Save readable JSON inside the project.
    """

    destination = PROJECT_ROOT / relative_path

    destination.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    return destination


print("Project root:")
print(PROJECT_ROOT)

print("\nPython version:")
print(sys.version.split()[0])

Project root:
/home/e9b07247-894a-49af-bfb3-922e50af0dcf

Python version:
3.11.17


# Create the asynchronous request dataset

In [2]:
# DAY 27 — SYNTHETIC ASYNC REQUESTS
#
# Purpose:
# Create five requests covering:
#
# - Two duplicate requests sharing one idempotency key
# - Two normal requests
# - One human-approval request
# - One request that exceeds the timeout
#
# delay_ms simulates provider response time.

async_requests = [
    {
        "request_id": "D27-R001",
        "idempotency_key": "journal-guidance-001",
        "user_role": "finance_analyst",
        "business_unit": "NA",
        "question": (
            "What approval is required for a "
            "USD 125,000 journal entry?"
        ),
        "delay_ms": 40,
    },
    {
        "request_id": "D27-R002",
        "idempotency_key": "journal-guidance-001",
        "user_role": "finance_analyst",
        "business_unit": "NA",
        "question": (
            "What approval is required for a "
            "USD 125,000 journal entry?"
        ),
        "delay_ms": 40,
    },
    {
        "request_id": "D27-R003",
        "idempotency_key": "period-close-guidance-001",
        "user_role": "controller",
        "business_unit": "EMEA",
        "question": (
            "What approval is required for a "
            "USD 75,000 period-end adjustment?"
        ),
        "delay_ms": 70,
    },
    {
        "request_id": "D27-R004",
        "idempotency_key": "journal-post-action-001",
        "user_role": "finance_analyst",
        "business_unit": "NA",
        "question": (
            "Post a USD 125,000 journal entry now."
        ),
        "delay_ms": 30,
    },
    {
        "request_id": "D27-R005",
        "idempotency_key": "slow-provider-001",
        "user_role": "auditor",
        "business_unit": "NA",
        "question": (
            "What is the journal-entry approval threshold?"
        ),
        "delay_ms": 250,
    },
]


requests_path = save_json(
    "data/day27_async_requests.json",
    async_requests,
)

print("Async requests saved:")
print(requests_path)

print("\nIncoming requests:")
print(len(async_requests))

print("\nUnique idempotency keys:")
print(
    len(
        {
            request[
                "idempotency_key"
            ]
            for request in async_requests
        }
    )
)

Async requests saved:
/home/e9b07247-894a-49af-bfb3-922e50af0dcf/data/day27_async_requests.json

Incoming requests:
5

Unique idempotency keys:
4


# Build the asynchronous provider gateway

In [3]:
# DAY 27 — ASYNC PROVIDER GATEWAY
#
# Purpose:
# Create a reusable gateway that:
#
# - Shares duplicate in-flight requests
# - Caches successful results by idempotency key
# - Limits concurrent provider calls
# - Enforces a provider timeout
# - Produces privacy-conscious audit events
#
# Retry logic is deliberately excluded because it was covered earlier.

gateway_source = '''
"""Asynchronous gateway for governed Finance model requests."""

from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Awaitable, Callable
import asyncio
import hashlib
import json


class AsyncFinanceGateway:
    """Control asynchronous provider access."""

    def __init__(
        self,
        provider: Callable[[dict], Awaitable[dict]],
        timeout_seconds: float,
        max_concurrency: int,
        audit_path: Path,
    ):
        if timeout_seconds <= 0:
            raise ValueError(
                "timeout_seconds must be greater than zero."
            )

        if max_concurrency <= 0:
            raise ValueError(
                "max_concurrency must be greater than zero."
            )

        self.provider = provider
        self.timeout_seconds = timeout_seconds
        self.audit_path = Path(audit_path)

        self._semaphore = asyncio.Semaphore(
            max_concurrency
        )

        self._state_lock = asyncio.Lock()
        self._audit_lock = asyncio.Lock()

        self._completed_results = {}
        self._inflight_tasks = {}

    @property
    def cache_size(self) -> int:
        """Return the number of successful cached operations."""

        return len(self._completed_results)

    @property
    def inflight_count(self) -> int:
        """Return the current number of in-flight operations."""

        return len(self._inflight_tasks)

    @staticmethod
    def _hash_key(idempotency_key: str) -> str:
        """Hash an idempotency key before audit logging."""

        return hashlib.sha256(
            idempotency_key.encode("utf-8")
        ).hexdigest()

    async def _write_audit(
        self,
        event_type: str,
        request: dict,
        **metadata,
    ) -> None:
        """
        Append one audit event.

        The raw question and raw idempotency key are not stored.
        """

        event = {
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "event_type": event_type,
            "request_id": request.get(
                "request_id"
            ),
            "user_role": request.get(
                "user_role"
            ),
            "business_unit": request.get(
                "business_unit"
            ),
            "idempotency_key_sha256": self._hash_key(
                request["idempotency_key"]
            ),
            **metadata,
        }

        self.audit_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        async with self._audit_lock:
            with self.audit_path.open(
                "a",
                encoding="utf-8",
            ) as audit_file:
                audit_file.write(
                    json.dumps(event)
                    + "\\n"
                )

    async def _execute_provider(
        self,
        request: dict,
    ) -> dict:
        """
        Execute one unique provider request with a timeout and
        concurrency limit.
        """

        started_at = perf_counter()

        async with self._semaphore:
            try:
                provider_result = await asyncio.wait_for(
                    self.provider(request),
                    timeout=self.timeout_seconds,
                )

                result = {
                    **provider_result,
                    "status": "success",
                }

            except asyncio.TimeoutError:
                result = {
                    "status": "timeout",
                    "answer": (
                        "The provider did not respond within "
                        "the configured time limit."
                    ),
                    "decision": "refuse",
                    "requires_human_approval": False,
                    "citations": [],
                    "rationale": (
                        "No Finance answer is produced after "
                        "a provider timeout."
                    ),
                }

            except Exception as error:
                result = {
                    "status": "provider_error",
                    "answer": (
                        "The provider request could not be completed."
                    ),
                    "decision": "refuse",
                    "requires_human_approval": False,
                    "citations": [],
                    "rationale": type(error).__name__,
                }

        elapsed_ms = round(
            (
                perf_counter()
                - started_at
            )
            * 1000,
            3,
        )

        result["latency_ms"] = elapsed_ms

        await self._write_audit(
            event_type="provider_request_completed",
            request=request,
            status=result["status"],
            decision=result["decision"],
            citations=result["citations"],
            requires_human_approval=result[
                "requires_human_approval"
            ],
            latency_ms=elapsed_ms,
        )

        return result

    async def _run_and_cache(
        self,
        request: dict,
    ) -> dict:
        """
        Run one unique request and cache only a successful result.
        """

        idempotency_key = request[
            "idempotency_key"
        ]

        try:
            result = await self._execute_provider(
                request
            )

            if result["status"] == "success":
                async with self._state_lock:
                    self._completed_results[
                        idempotency_key
                    ] = deepcopy(result)

            return result

        finally:
            async with self._state_lock:
                self._inflight_tasks.pop(
                    idempotency_key,
                    None,
                )

    async def process(
        self,
        request: dict,
    ) -> dict:
        """
        Process one request with idempotency protection.
        """

        idempotency_key = request.get(
            "idempotency_key"
        )

        if not idempotency_key:
            raise ValueError(
                "Every request requires an idempotency_key."
            )

        audit_event = None

        async with self._state_lock:
            if (
                idempotency_key
                in self._completed_results
            ):
                cached_result = deepcopy(
                    self._completed_results[
                        idempotency_key
                    ]
                )

                audit_event = (
                    "idempotency_cache_hit"
                )

                task = None
                reused = True

            else:
                task = self._inflight_tasks.get(
                    idempotency_key
                )

                if task is None:
                    task = asyncio.create_task(
                        self._run_and_cache(
                            request
                        )
                    )

                    self._inflight_tasks[
                        idempotency_key
                    ] = task

                    audit_event = (
                        "idempotency_operation_started"
                    )

                    reused = False

                else:
                    audit_event = (
                        "idempotency_inflight_joined"
                    )

                    reused = True

        await self._write_audit(
            event_type=audit_event,
            request=request,
        )

        if task is None:
            result = cached_result
        else:
            result = await task

        return {
            "request_id": request[
                "request_id"
            ],
            "idempotency_reused": reused,
            "result": deepcopy(result),
        }

    async def process_batch(
        self,
        requests: list[dict],
    ) -> list[dict]:
        """
        Process several requests concurrently while preserving
        their original list order.
        """

        return await asyncio.gather(
            *(
                self.process(request)
                for request in requests
            )
        )
'''

gateway_path = (
    PROJECT_ROOT
    / "app"
    / "async_finance_gateway.py"
)

gateway_path.write_text(
    gateway_source.strip() + "\n",
    encoding="utf-8",
)

print("Async gateway created:")
print(gateway_path)

Async gateway created:
/home/e9b07247-894a-49af-bfb3-922e50af0dcf/app/async_finance_gateway.py


# Simulate concurrent provider requests

In [4]:
# DAY 27 — RUN THE ASYNC GATEWAY
#
# Purpose:
# Use a deterministic provider simulation to verify concurrency,
# duplicate suppression, controlled escalation and timeout behavior.
#
# This makes no external model or network call.

module_spec = importlib.util.spec_from_file_location(
    "async_finance_gateway",
    gateway_path,
)

gateway_module = (
    importlib.util.module_from_spec(
        module_spec
    )
)

module_spec.loader.exec_module(
    gateway_module
)


provider_statistics = {
    "active_calls": 0,
    "maximum_active_calls": 0,
    "calls_by_idempotency_key": {},
}


async def simulated_finance_provider(
    request: dict,
) -> dict:
    """
    Simulate an asynchronous model provider.

    delay_ms controls how long the simulated provider takes.
    """

    idempotency_key = request[
        "idempotency_key"
    ]

    provider_statistics[
        "calls_by_idempotency_key"
    ][idempotency_key] = (
        provider_statistics[
            "calls_by_idempotency_key"
        ].get(idempotency_key, 0)
        + 1
    )

    provider_statistics[
        "active_calls"
    ] += 1

    provider_statistics[
        "maximum_active_calls"
    ] = max(
        provider_statistics[
            "maximum_active_calls"
        ],
        provider_statistics[
            "active_calls"
        ],
    )

    try:
        await asyncio.sleep(
            request["delay_ms"] / 1000
        )

        normalized_question = request[
            "question"
        ].lower()

        action_requested = any(
            action_word in normalized_question
            for action_word in [
                "post",
                "approve",
                "reverse",
                "modify",
                "execute",
            ]
        )

        if action_requested:
            return {
                "answer": (
                    "I cannot execute this financial transaction. "
                    "A human approver must review it."
                ),
                "decision": "escalate",
                "requires_human_approval": True,
                "citations": ["FIN-JE-001"],
                "rationale": (
                    "Financial execution is outside the "
                    "assistant's authority."
                ),
            }

        if (
            "period-end" in normalized_question
            or "adjustment" in normalized_question
        ):
            return {
                "answer": (
                    "Adjustments above USD 50,000 require "
                    "a regional finance manager."
                ),
                "decision": "answer",
                "requires_human_approval": False,
                "citations": ["FIN-CLOSE-001"],
                "rationale": (
                    "The authorized closing policy supports "
                    "the response."
                ),
            }

        if "journal" in normalized_question:
            return {
                "answer": (
                    "Journal entries above USD 100,000 require "
                    "approval by a finance manager."
                ),
                "decision": "answer",
                "requires_human_approval": False,
                "citations": ["FIN-JE-001"],
                "rationale": (
                    "The authorized journal policy supports "
                    "the response."
                ),
            }

        return {
            "answer": (
                "No authorized policy supports this request."
            ),
            "decision": "refuse",
            "requires_human_approval": False,
            "citations": [],
            "rationale": (
                "The available context is insufficient."
            ),
        }

    finally:
        provider_statistics[
            "active_calls"
        ] -= 1


audit_path = (
    PROJECT_ROOT
    / "audit"
    / "day27_async_gateway_audit.jsonl"
)

# Clear only today's generated audit file when rerunning this notebook.
audit_path.write_text(
    "",
    encoding="utf-8",
)


gateway = gateway_module.AsyncFinanceGateway(
    provider=simulated_finance_provider,
    timeout_seconds=0.10,
    max_concurrency=2,
    audit_path=audit_path,
)


batch_results = await gateway.process_batch(
    async_requests
)


print("Async request results:\n")

for response in batch_results:
    result = response["result"]

    print(
        response["request_id"],
        "| status:",
        result["status"],
        "| decision:",
        result["decision"],
        "| reused:",
        response["idempotency_reused"],
        "| latency:",
        result["latency_ms"],
        "ms",
    )

print("\nProvider calls by idempotency key:")
print(
    provider_statistics[
        "calls_by_idempotency_key"
    ]
)

print("\nMaximum simultaneous provider calls:")
print(
    provider_statistics[
        "maximum_active_calls"
    ]
)

Async request results:

D27-R001 | status: success | decision: answer | reused: False | latency: 40.299 ms
D27-R002 | status: success | decision: answer | reused: True | latency: 40.299 ms
D27-R003 | status: success | decision: answer | reused: False | latency: 71.187 ms
D27-R004 | status: success | decision: escalate | reused: False | latency: 74.789 ms
D27-R005 | status: timeout | decision: refuse | reused: False | latency: 175.536 ms

Provider calls by idempotency key:
{'journal-guidance-001': 1, 'period-close-guidance-001': 1, 'journal-post-action-001': 1, 'slow-provider-001': 1}

Maximum simultaneous provider calls:
2


# Create the resilience report

In [5]:
# DAY 27 — ASYNC RESILIENCE REPORT
#
# Purpose:
# Measure idempotency reuse, provider-call reduction, concurrency,
# timeout handling and human-approval behavior.

results_by_request = {
    response["request_id"]: response
    for response in batch_results
}

unique_provider_calls = sum(
    provider_statistics[
        "calls_by_idempotency_key"
    ].values()
)

duplicate_responses = [
    response
    for response in batch_results
    if response["request_id"]
    in {
        "D27-R001",
        "D27-R002",
    }
]

resilience_checks = {
    "five_responses_returned": (
        len(batch_results) == 5
    ),
    "duplicate_used_one_provider_call": (
        provider_statistics[
            "calls_by_idempotency_key"
        ]["journal-guidance-001"]
        == 1
    ),
    "one_duplicate_marked_reused": (
        sum(
            response[
                "idempotency_reused"
            ]
            for response in duplicate_responses
        )
        == 1
    ),
    "provider_calls_reduced_to_four": (
        unique_provider_calls == 4
    ),
    "concurrency_limit_respected": (
        provider_statistics[
            "maximum_active_calls"
        ]
        <= 2
    ),
    "slow_request_timed_out": (
        results_by_request[
            "D27-R005"
        ]["result"]["status"]
        == "timeout"
    ),
    "execution_request_escalated": (
        results_by_request[
            "D27-R004"
        ]["result"]["decision"]
        == "escalate"
        and results_by_request[
            "D27-R004"
        ]["result"][
            "requires_human_approval"
        ]
        is True
    ),
    "no_inflight_tasks_remain": (
        gateway.inflight_count == 0
    ),
}


resilience_report = {
    "report_id": (
        "day27-async-api-resilience"
    ),
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "configuration": {
        "timeout_seconds": (
            gateway.timeout_seconds
        ),
        "max_concurrency": 2,
    },
    "metrics": {
        "incoming_requests": len(
            async_requests
        ),
        "unique_idempotency_keys": len(
            {
                request[
                    "idempotency_key"
                ]
                for request in async_requests
            }
        ),
        "provider_calls": unique_provider_calls,
        "provider_calls_avoided": (
            len(async_requests)
            - unique_provider_calls
        ),
        "maximum_active_provider_calls": (
            provider_statistics[
                "maximum_active_calls"
            ]
        ),
        "successful_cache_entries": (
            gateway.cache_size
        ),
        "timeout_count": sum(
            response["result"]["status"]
            == "timeout"
            for response in batch_results
        ),
    },
    "checks": resilience_checks,
    "responses": batch_results,
}


report_path = save_json(
    "artifacts/day27_async_gateway_report.json",
    resilience_report,
)


print("Day 27 resilience checks:\n")

for check_name, check_passed in resilience_checks.items():
    icon = "✅" if check_passed else "❌"
    print(f"{icon} {check_name}")

print("\nMetrics:")
print(
    json.dumps(
        resilience_report["metrics"],
        indent=2,
    )
)

print("\nReport saved:")
print(report_path)

Day 27 resilience checks:

✅ five_responses_returned
✅ duplicate_used_one_provider_call
✅ one_duplicate_marked_reused
✅ provider_calls_reduced_to_four
✅ concurrency_limit_respected
✅ slow_request_timed_out
✅ execution_request_escalated
✅ no_inflight_tasks_remain

Metrics:
{
  "incoming_requests": 5,
  "unique_idempotency_keys": 4,
  "provider_calls": 4,
  "provider_calls_avoided": 1,
  "maximum_active_provider_calls": 2,
  "successful_cache_entries": 3,
  "timeout_count": 1
}

Report saved:
/home/e9b07247-894a-49af-bfb3-922e50af0dcf/artifacts/day27_async_gateway_report.json


# Write the engineering notes

In [6]:
# DAY 27 — DOCUMENTATION
#
# Purpose:
# Record what the gateway protects, how it maps to enterprise APIs,
# and what must change before production deployment.

notes_text = f"""# Day 27 — Async API Resilience and Idempotency

## Objective

Add asynchronous request handling, idempotency protection, provider
timeouts and concurrency limits to the governed SAP Finance assistant.

## Results

- Incoming requests: {resilience_report["metrics"]["incoming_requests"]}
- Unique idempotency keys: {resilience_report["metrics"]["unique_idempotency_keys"]}
- Provider calls executed: {resilience_report["metrics"]["provider_calls"]}
- Duplicate provider calls avoided: {resilience_report["metrics"]["provider_calls_avoided"]}
- Maximum concurrent calls: {resilience_report["metrics"]["maximum_active_provider_calls"]}
- Timeout responses: {resilience_report["metrics"]["timeout_count"]}

## Controls implemented

1. Duplicate requests with the same idempotency key share one in-flight task.
2. Successful results are cached for later duplicate requests.
3. Timed-out requests are not cached.
4. A semaphore limits simultaneous provider usage.
5. Slow providers return a controlled refusal rather than an invented answer.
6. Transaction execution continues to require human approval.
7. Audit events contain a hashed idempotency key and no raw question.

## Databricks alignment

The gateway models client-side controls needed when calling model-serving
or agent endpoints: bounded concurrency, timeout handling, safe repetition
and latency measurement.

## SAP alignment

SAP Cloud SDK for AI supports asynchronous orchestration calls and request
timeouts. The simulated provider can later be replaced by an SAP
orchestration `arun` call without changing the gateway contract.

## Production limitation

The cache is currently in memory and is local to one Python process.
A scaled SAP BTP or Databricks deployment would need a shared persistent
idempotency store such as Redis, a database or a governed Delta table.

## Interview explanation

I prevented duplicate Finance requests from generating duplicate provider
calls by assigning each logical operation an idempotency key. Concurrent
duplicates join the same in-flight task, provider concurrency is bounded,
and slow requests fail safely without generating unsupported guidance.
"""


notes_path = (
    PROJECT_ROOT
    / "notes"
    / "day27_python_api_resilience.md"
)

notes_path.write_text(
    notes_text,
    encoding="utf-8",
)

print("Engineering notes created:")
print(notes_path)

Engineering notes created:
/home/e9b07247-894a-49af-bfb3-922e50af0dcf/notes/day27_python_api_resilience.md


# DAY 27 — COMMIT AND SECURELY PUSH TO GITHUB

from pathlib import Path
from getpass import getpass
import os
import shutil
import subprocess
import tempfile


starting_folder = Path.cwd().resolve()


# ---------------------------------------------------------
# 1. LOCATE THE CORRECT GIT REPOSITORY
# ---------------------------------------------------------

repository_candidates = [
    starting_folder,
    *starting_folder.parents,
]

projects_area = Path(
    "/var/www/filebrowser/.projects"
)

if projects_area.exists():
    repository_candidates.extend(
        projects_area.glob(
            "*/sap-finance-ai-assistant"
        )
    )


valid_repositories = [
    path.resolve()
    for path in repository_candidates
    if path.exists()
    and (path / ".git").exists()
]


if not valid_repositories:
    raise FileNotFoundError(
        "The Git repository could not be located."
    )


GIT_ROOT = max(
    valid_repositories,
    key=lambda path: (
        path
        / "artifacts"
        / "day27_async_gateway_report.json"
    ).exists(),
)


os.chdir(GIT_ROOT)

print("Git repository:")
print(GIT_ROOT)


# ---------------------------------------------------------
# 2. RECOVER THE NOTEBOOK IF NECESSARY
# ---------------------------------------------------------

notebook_name = (
    "day27_async_api_resilience.ipynb"
)

notebook_destination = (
    GIT_ROOT
    / "notebooks"
    / notebook_name
)


if not notebook_destination.exists():
    notebook_candidates = [
        starting_folder / notebook_name,
        Path.home() / notebook_name,
        (
            Path.home()
            / "notebooks"
            / notebook_name
        ),
        GIT_ROOT / notebook_name,
    ]

    try:
        notebook_candidates.extend(
            Path.home().rglob(
                notebook_name
            )
        )
    except OSError:
        pass

    notebook_sources = [
        candidate.resolve()
        for candidate in notebook_candidates
        if candidate.exists()
        and ".ipynb_checkpoints"
        not in candidate.parts
        and candidate.resolve()
        != notebook_destination.resolve()
    ]

    if not notebook_sources:
        raise FileNotFoundError(
            "Save the notebook as "
            f"{notebook_name}, then rerun this cell."
        )

    notebook_source = max(
        notebook_sources,
        key=lambda path: (
            path.stat().st_mtime
        ),
    )

    notebook_destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        notebook_source,
        notebook_destination,
    )

    print(
        "\n✅ Notebook copied to:",
        notebook_destination,
    )

else:
    print(
        "\n✅ Notebook found:",
        notebook_destination.relative_to(
            GIT_ROOT
        ),
    )


# ---------------------------------------------------------
# 3. VERIFY DAY 27 FILES
# ---------------------------------------------------------

day27_files = [
    "data/day27_async_requests.json",
    "app/async_finance_gateway.py",
    "artifacts/day27_async_gateway_report.json",
    "audit/day27_async_gateway_audit.jsonl",
    "notes/day27_python_api_resilience.md",
    "notebooks/day27_async_api_resilience.ipynb",
]


missing_files = [
    file_name
    for file_name in day27_files
    if not (
        GIT_ROOT / file_name
    ).exists()
]


if missing_files:
    print("\nMissing Day 27 files:")

    for file_name in missing_files:
        print("❌", file_name)

    raise FileNotFoundError(
        "Complete the missing Day 27 outputs "
        "before committing."
    )


print("\nDay 27 files ready:")

for file_name in day27_files:
    print("✅", file_name)


# ---------------------------------------------------------
# 4. GIT HELPER
# ---------------------------------------------------------

def run_git(*arguments):
    result = subprocess.run(
        ["git", *arguments],
        cwd=GIT_ROOT,
        text=True,
        capture_output=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip()
            or result.stdout.strip()
        )

    return result.stdout.strip()


branch_name = run_git(
    "branch",
    "--show-current",
)


if branch_name != "main":
    raise RuntimeError(
        f"Expected branch main, found {branch_name}."
    )


# Prevent unrelated staged files from entering this commit.
already_staged = run_git(
    "diff",
    "--cached",
    "--name-only",
)


unexpected_staged = [
    file_name
    for file_name in already_staged.splitlines()
    if file_name
    and file_name not in day27_files
]


if unexpected_staged:
    raise RuntimeError(
        "Unrelated files are already staged: "
        f"{unexpected_staged}"
    )


# ---------------------------------------------------------
# 5. STAGE AND COMMIT
# ---------------------------------------------------------

run_git(
    "add",
    "--",
    *day27_files,
)


staged_files = run_git(
    "diff",
    "--cached",
    "--name-only",
)


if staged_files:
    print("\nFiles being committed:")
    print(staged_files)

    commit_output = run_git(
        "commit",
        "-m",
        (
            "Add async provider gateway "
            "and idempotency controls"
        ),
    )

    print("\n" + commit_output)

else:
    print(
        "\nNo new Day 27 changes require a commit. "
        "They may already be committed."
    )


print("\nLatest local commit:")
print(
    run_git(
        "log",
        "-1",
        "--oneline",
    )
)


# ---------------------------------------------------------
# 6. SECURE GITHUB AUTHENTICATION
# ---------------------------------------------------------

github_username = input(
    "\nGitHub username: "
).strip()

github_token = getpass(
    "GitHub personal access token (hidden): "
)


if not github_username:
    raise ValueError(
        "GitHub username is required."
    )

if not github_token:
    raise ValueError(
        "GitHub token is required."
    )


with tempfile.TemporaryDirectory() as temporary_folder:
    askpass_file = (
        Path(temporary_folder)
        / "git_askpass.sh"
    )

    askpass_file.write_text(
        """#!/bin/sh
case "$1" in
    *sername*) printf '%s\\n' "$GITHUB_USERNAME" ;;
    *assword*) printf '%s\\n' "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
    )

    askpass_file.chmod(0o700)

    secure_environment = os.environ.copy()

    secure_environment.update(
        {
            "GIT_ASKPASS": str(askpass_file),
            "GIT_TERMINAL_PROMPT": "0",
            "GITHUB_USERNAME": github_username,
            "GITHUB_TOKEN": github_token,
        }
    )

    push_result = subprocess.run(
        [
            "git",
            "push",
            "origin",
            "HEAD:main",
        ],
        cwd=GIT_ROOT,
        env=secure_environment,
        text=True,
        capture_output=True,
    )

    verification_result = subprocess.run(
        [
            "git",
            "ls-remote",
            "origin",
            "refs/heads/main",
        ],
        cwd=GIT_ROOT,
        env=secure_environment,
        text=True,
        capture_output=True,
    )


safe_output = (
    push_result.stdout
    + push_result.stderr
).replace(
    github_token,
    "***",
)


if push_result.returncode != 0:
    github_token = None

    raise RuntimeError(
        "GitHub push failed:\n"
        + safe_output.strip()
    )


if verification_result.returncode != 0:
    github_token = None

    raise RuntimeError(
        "The push completed, but origin/main "
        "could not be verified."
    )


local_commit = run_git(
    "rev-parse",
    "HEAD",
)

remote_commit = (
    verification_result
    .stdout
    .strip()
    .split()[0]
)


github_token = None


if local_commit != remote_commit:
    raise AssertionError(
        "Local HEAD does not match origin/main."
    )


print("\n" + safe_output.strip())

print(
    "\n✅ Day 27 committed and pushed to GitHub."
)

print(
    "✅ Local HEAD matches origin/main:",
    local_commit[:12],
)